# Settimana 5 — Bortkiewicz e i calci di cavallo: distribuzione geometrica e distribuzione di Poisson

Questo è il notebook della Settimana 5 del corso di Probabilità e Statistica. Contiene gli script Python della [dispensa della settimana](https://qwertyjacob.github.io/PSI-VA-2627/settimana_05/), nello stesso ordine in cui compaiono: ogni script è una cella di codice, preceduta dal titolo della sezione in cui si trova. Le spiegazioni restano nella dispensa.

Per eseguire una cella, premi il pulsante ▶ alla sua sinistra. Gli script sono indipendenti l'uno dall'altro, quindi puoi eseguirli in qualsiasi ordine e modificarli liberamente. Colab non salva le modifiche nel notebook originale: se vuoi conservarle, scegli *File → Salva una copia in Drive*.

## 7. Laboratorio: osservare molte attese

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_05/#lab-geometrica)

In [ ]:
from random import Random

generatore = Random(42)
p = 1 / 18          # il 53 è uno dei 5 numeri estratti su 90
prove = 50000       # quante attese complete simuliamo


def attesa(p):
    """Numero dell'estrazione in cui arriva il primo successo."""
    k = 1
    # Ogni estrazione è nuova: la probabilità non dipende dal passato.
    while generatore.random() >= p:
        k += 1
    return k


attese = [attesa(p) for _ in range(prove)]
print(f"Media teorica 1/p: {1 / p:.2f}")
print(f"Media simulata: {sum(attese) / prove:.2f}")

# Coda: più di 18 estrazioni significa 18 fallimenti di fila.
oltre_18 = sum(x > 18 for x in attese) / prove
print(f"P(X > 18) teorica: {(1 - p) ** 18:.4f}")
print(f"P(X > 18) simulata: {oltre_18:.4f}")

# Assenza di memoria: guardiamo solo le attese già oltre 30.
lunghe = [x for x in attese if x > 30]
ancora = sum(x > 30 + 18 for x in lunghe) / len(lunghe)
print(f"Attese già oltre 30: {len(lunghe)}")
print(f"Tra queste, ancora 18 fallimenti: {ancora:.4f}")

## 16. Laboratorio: ricostruire tabella e dispersione

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_05/#lab-bortkiewicz)

In [ ]:
from math import exp, factorial

import matplotlib.pyplot as plt

# Bortkiewicz (1898), § 12: 10 corpi d'armata per 20 anni.
osservati = {0: 109, 1: 65, 2: 22, 3: 3, 4: 1}
corpi_anno = sum(osservati.values())
morti = sum(k * f for k, f in osservati.items())
lam = morti / corpi_anno   # numero medio di morti per corpo-anno
assert corpi_anno == 200 and morti == 122


def poisson(k, lam):
    return exp(-lam) * lam ** k / factorial(k)


print(f"lambda = {morti}/{corpi_anno} = {lam:.2f}")
print(" k  osservati  attesi")
for k, f in osservati.items():
    # Conteggio atteso = numero di corpi-anno x probabilità.
    print(f"{k:2d}  {f:9d}  {corpi_anno * poisson(k, lam):6.1f}")

# Dispersione descrittiva: la varianza usa qui il divisore 200.
media = morti / corpi_anno
varianza = sum((k - media) ** 2 * f
               for k, f in osservati.items()) / corpi_anno
print(f"Media dei dati: {media:.3f}")
print(f"Varianza dei dati: {varianza:.3f}")

k = list(osservati)
plt.bar([x - 0.2 for x in k], list(osservati.values()), width=0.4,
        label="Osservati (Bortkiewicz)")
plt.bar([x + 0.2 for x in k],
        [corpi_anno * poisson(x, lam) for x in k], width=0.4,
        label="Attesi con la distribuzione di Poisson(0,61)")
plt.xticks(k)
plt.xlabel("Morti in un corpo-anno")
plt.ylabel("Numero di corpi-anno")
plt.legend()
plt.tight_layout()
plt.show()

## 20. Laboratorio: traffico regolare e traffico a raffiche

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_05/#lab-richieste)

In [ ]:
from math import exp, factorial

import numpy as np

rng = np.random.default_rng(42)
utenti = 10000      # client che possono inviare una richiesta
p = 0.0003          # probabilità che uno di loro la invii in 1 secondo
secondi = 20000     # quanti secondi osserviamo
raffica = False     # prova True: a volte tutti diventano più attivi

lam = utenti * p    # richieste attese al secondo
p_secondo = np.full(secondi, p)
if raffica:
    # Nel 10% dei secondi ogni client è 5 volte più attivo.
    p_secondo[rng.random(secondi) < 0.1] = 5 * p
# Ogni secondo: quanti dei 10000 client inviano (una distribuzione binomiale).
richieste = rng.binomial(utenti, p_secondo)

media = richieste.mean()
print(f"Media di base n * p: {lam:.1f}")
media_teorica = lam * (1.4 if raffica else 1)
print(f"Media teorica del modello scelto: {media_teorica:.1f}")
print(f"Media osservata: {media:.3f}")
print(f"Varianza osservata: {richieste.var():.3f}")
# Confronto con la distribuzione di Poisson che ha la stessa media dei dati.
print(" k  frequenza  distribuzione di Poisson")
for k in range(9):
    freq = np.mean(richieste == k)
    teoria = exp(-media) * media ** k / factorial(k)
    print(f"{k:2d}  {freq:9.4f}  {teoria:7.4f}")